# Bonus Topic 1 - Natural Language Processing with NLTK and spaCy

This notebook introduces practical Natural Language Processing (NLP) techniques in Python using English examples.

We will use two libraries:

- **NLTK** for classic NLP techniques, corpora, linguistic resources, and transparent teaching examples.
- **spaCy** for modern pipeline-based NLP, named entity recognition, dependency parsing, phrase extraction, and rule-based matching.

The notebook is designed to run in a local Jupyter environment and in **Google Colab**. Run the setup cells first when using a fresh environment.


## Lesson Overview

Natural Language Processing is the area of computing concerned with representing, processing, analyzing, and generating human language.

In this notebook we will follow a practical workflow:

1. Start with raw English text.
2. Split it into sentences and tokens.
3. Normalize tokens with lowercasing, stemming, or lemmatization.
4. Add linguistic annotations such as part-of-speech tags, dependency relations, and named entities.
5. Extract useful information with counting, rules, and simple classification.
6. Compare NLTK and spaCy on similar tasks.

The goal is not to cover every modern NLP method. Transformers, embeddings, and large language models are important next topics, but this notebook focuses on core techniques students can understand directly from Python data structures.


## Learning Objectives

By the end of this notebook, you should be able to:

- explain tokenization, sentence segmentation, stop words, stemming, lemmatization, POS tagging, named entity recognition, dependency parsing, and noun chunks;
- use NLTK to tokenize text, remove stop words, count words, stem, lemmatize, tag parts of speech, chunk noun phrases, extract named entities, use WordNet, and train a tiny classifier;
- use spaCy to load an English pipeline and inspect `Doc`, `Token`, and `Span` objects;
- use spaCy for lemmas, POS tags, dependencies, noun chunks, named entities, visualization, rule-based matching, and batch processing;
- choose between NLTK and spaCy based on the task.


## Lesson Prerequisites

You should already be comfortable with:

- Python strings;
- lists, tuples, dictionaries, and sets;
- loops and comprehensions;
- functions;
- installing packages with `pip`;
- basic Jupyter notebook usage.

No previous NLP experience is required.


## Google Colab and Local Installation

If you are using **Google Colab**, run the next setup cell. It installs missing Python packages and downloads the small English spaCy pipeline.

If you are using a local environment, the same cell also works, but it may install packages into the currently selected Jupyter kernel. If you prefer manual setup, use these commands in your activated virtual environment:

```bash
pip install nltk spacy
python -m spacy download en_core_web_sm
```

The notebook also downloads NLTK data packages in a later setup cell. NLTK separates the Python package from many linguistic resources, so both steps are needed.


In [ ]:
# Colab/local setup cell.
# Run this first in Google Colab or in a fresh local Jupyter environment.

import importlib.util
import subprocess
import sys


def ensure_import(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    else:
        print(f"{import_name} is already available.")


ensure_import("nltk")
ensure_import("spacy")

import nltk
import spacy

try:
    spacy.load("en_core_web_sm")
    print("spaCy English pipeline en_core_web_sm is already available.")
except OSError:
    print("Downloading spaCy English pipeline en_core_web_sm...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

print("Setup complete.")


## NLTK Data Downloads

NLTK functions often require separate data packages. The next cell downloads the packages used in this notebook.

The list includes current NLTK 3.9 resource names such as `punkt_tab`, `averaged_perceptron_tagger_eng`, and `maxent_ne_chunker_tab`. It also includes older names such as `punkt`, `averaged_perceptron_tagger`, and `maxent_ne_chunker` for compatibility with older notebooks and environments.

If a later cell raises `LookupError`, read the error message and download the exact resource name it requests.


In [ ]:
import nltk

nltk_packages = [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words",
    "wordnet",
    "omw-1.4",
    "stopwords",
    "universal_tagset",
    "brown",
]

for package in nltk_packages:
    try:
        ok = nltk.download(package, quiet=True)
        status = "ok" if ok else "not found or already unavailable in this NLTK index"
        print(f"{package}: {status}")
    except Exception as exc:
        print(f"{package}: {exc}")


## Version Check

NLP output can change when package versions or model versions change. For reproducible work, record the package versions and the spaCy pipeline version.


In [ ]:
import nltk
import spacy

nlp = spacy.load("en_core_web_sm")

print("NLTK version:", nltk.__version__)
print("spaCy version:", spacy.__version__)
print("spaCy pipeline:", nlp.meta.get("name"), nlp.meta.get("version"))
print("Pipeline components:", nlp.pipe_names)


## Example Texts

We will use small English examples so the notebook does not depend on external files.


In [ ]:
main_text = (
    "In 2026, Anna joined Acme Robotics in London. "
    "She builds Python tools for language data, and she enjoys teaching Python."
)

review_text = (
    "This Python notebook is practical and clear, but the installation steps are a little long. "
    "Overall, I would recommend the course to beginners."
)

news_text = (
    "Marie Curie worked in Paris and won the Nobel Prize. "
    "Google announced a new research office in Berlin on Monday."
)

email_text = (
    "Dear team, please send the final NLP report to Anna before Friday. "
    "The client from London expects a short summary."
)

print(main_text)


## NLP Pipeline

A typical NLP pipeline converts raw strings into structured information.

```text
Raw text
  -> sentence segmentation
  -> tokenization
  -> normalization
  -> linguistic annotation
  -> extraction, counting, classification, or search
```

The same text can be represented in many ways: as characters, sentences, tokens, lemmas, POS tags, named entities, dependency trees, or feature dictionaries.


## Essential Terminology

| Term | Practical meaning | Example |
| --- | --- | --- |
| Corpus | A collection of texts used for analysis or training | product reviews, news articles, books |
| Document | One text unit in a corpus | one review, one email, one article |
| Sentence segmentation | Splitting text into sentences | `Hello. I am here.` -> two sentences |
| Tokenization | Splitting text into meaningful pieces | `don't` may become `do` + `n't` |
| Token | One processed unit of text | word, punctuation mark, number |
| Normalization | Making forms more comparable | lowercasing, removing punctuation, lemmatizing |
| Stop word | Very common word sometimes removed for search or counting | `the`, `and`, `of` |
| Stem | A reduced word form, often produced by heuristic suffix removal | `running` -> `run` or `runn` |
| Lemma | Dictionary or base form | `was` -> `be`, `cars` -> `car` |
| POS tag | Part-of-speech label | noun, verb, adjective |
| Named entity | Named real-world object or value | person, organization, place, date |
| Dependency parse | Grammatical links between words | subject, object, modifier |
| Feature | Input signal used by a classifier | whether a word appears in a document |


# Part 1: NLTK

NLTK is useful for teaching classic NLP concepts because each step is visible as a separate function call or data structure.

Typical NLTK output types include:

- lists of strings;
- lists of `(word, tag)` tuples;
- `FreqDist` frequency distributions;
- NLTK trees;
- feature dictionaries for classifiers.


## NLTK: Sentence and Word Tokenization

Tokenization splits text into smaller units. Sentence tokenization and word tokenization are different tasks.


In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

sentences = sent_tokenize(main_text)
tokens = word_tokenize(main_text)

print("Sentences:")
for sentence in sentences:
    print("-", sentence)

print("\nTokens:")
print(tokens)


### Why not just use `split()`?

Python's `str.split()` only splits on whitespace by default. NLP tokenizers handle punctuation and some language-specific rules.


In [ ]:
simple_split = main_text.split()
nltk_tokens = word_tokenize(main_text)

print("str.split():")
print(simple_split[:15])

print("\nNLTK word_tokenize():")
print(nltk_tokens[:20])


## NLTK: Regular Expression Tokenization

For some tasks, we want only alphabetic word tokens. NLTK provides tokenizer classes for custom tokenization.


In [ ]:
from nltk.tokenize import RegexpTokenizer

alpha_tokenizer = RegexpTokenizer(r"[A-Za-z]+")
alpha_tokens = alpha_tokenizer.tokenize(main_text)

print(alpha_tokens)


## NLTK: Lowercasing and Stop Words

Lowercasing is useful for counting, but it can remove meaningful differences such as `Apple` the company versus `apple` the fruit.

Stop words are common words that are sometimes removed before counting or indexing. This is useful for simple keyword frequency examples, but it is not a universal rule.


In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

filtered_tokens = [
    token.lower()
    for token in word_tokenize(review_text)
    if token.isalpha() and token.lower() not in stop_words
]

print("Original tokens:")
print(word_tokenize(review_text))

print("\nFiltered content words:")
print(filtered_tokens)


### Stop Word Caution

Removing stop words can damage meaning. For example, `not good` and `good` are very different, but `not` is often included in stop word lists.


In [ ]:
print("Is 'not' an English stop word in NLTK?", "not" in stop_words)

sentiment_example = "The explanation is not clear."
print(word_tokenize(sentiment_example))
print([
    token.lower()
    for token in word_tokenize(sentiment_example)
    if token.isalpha() and token.lower() not in stop_words
])


## NLTK: Frequency Analysis

`FreqDist` counts observed samples. It is useful for exploring word frequency in a text.


In [ ]:
from nltk.probability import FreqDist

review_words = [
    token.lower()
    for token in word_tokenize(review_text)
    if token.isalpha() and token.lower() not in stop_words
]

fdist = FreqDist(review_words)

print("Most common words:")
print(fdist.most_common(10))

print("Count for 'python':", fdist["python"])


## NLTK: Stemming

Stemming reduces words to stems, usually with heuristic suffix rules. Stems are not always dictionary words.


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

words = ["connect", "connected", "connecting", "connection", "connections", "studies", "studying"]

for word in words:
    print(f"{word:12} -> {stemmer.stem(word)}")


## NLTK: Lemmatization with WordNet

Lemmatization aims to return a dictionary form. NLTK's `WordNetLemmatizer` assumes nouns by default, so part of speech matters.


In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

words = ["cars", "running", "better", "was", "studies"]

print("Default noun lemmatization:")
for word in words:
    print(f"{word:10} -> {lemmatizer.lemmatize(word)}")

print("\nWith explicit part of speech:")
examples = [("running", "v"), ("better", "a"), ("was", "v"), ("studies", "v")]
for word, pos in examples:
    print(f"{word:10} ({pos}) -> {lemmatizer.lemmatize(word, pos=pos)}")


## NLTK: POS Tagging

Part-of-speech tagging assigns grammatical labels to tokens.

NLTK's default English POS tags use Penn Treebank-style labels such as `NN`, `NNP`, `VB`, `VBD`, and `JJ`.


In [ ]:
from nltk import pos_tag

sentence = "Anna builds Python tools for language data."
sentence_tokens = word_tokenize(sentence)
tagged = pos_tag(sentence_tokens)

print(tagged)


### Universal POS Tags

The universal tagset is smaller and easier to read for beginners.


In [ ]:
tagged_universal = pos_tag(sentence_tokens, tagset="universal")
print(tagged_universal)


## NLTK: Chunking

Chunking groups tagged tokens into shallow phrases. A common example is noun phrase chunking.


In [ ]:
import nltk

chunk_grammar = "NP: {<DT>?<JJ.*>*<NN.*>+}"

chunk_parser = nltk.RegexpParser(chunk_grammar)
chunk_tree = chunk_parser.parse(tagged)

print(chunk_tree)


## NLTK: Named Entity Chunking

Named Entity Recognition identifies names and other real-world references such as people, organizations, places, dates, and money amounts.

NLTK's named entity chunker returns an NLTK tree.


In [ ]:
from nltk.chunk import ne_chunk

entity_tokens = word_tokenize(news_text)
entity_tags = pos_tag(entity_tokens)
entity_tree = ne_chunk(entity_tags)

print(entity_tree)


In [ ]:
print("Extracted entity subtrees:")
for node in entity_tree:
    if hasattr(node, "label"):
        entity_text = " ".join(token for token, tag in node.leaves())
        print(f"{entity_text:20} -> {node.label()}")


## NLTK: WordNet

WordNet is a lexical database of English. It groups nouns, verbs, adjectives, and adverbs into sets of cognitive synonyms called synsets, and links those synsets through semantic relations.


In [ ]:
from nltk.corpus import wordnet as wn

for synset in wn.synsets("course")[:5]:
    print(synset.name(), "-", synset.definition())


In [ ]:
dog = wn.synset("dog.n.01")

print("Definition:", dog.definition())
print("Hypernyms:", dog.hypernyms())
print("First five hyponyms:", dog.hyponyms()[:5])


## NLTK: Concordance and Corpus Exploration

NLTK can wrap tokens in an `nltk.Text` object for classic corpus exploration.


In [ ]:
from nltk.text import Text

nltk_text = Text(word_tokenize(main_text))
nltk_text.concordance("Python")


NLTK also includes corpus readers. The Brown Corpus is a classic English corpus useful for demonstrations.


In [ ]:
from nltk.corpus import brown

brown_words = brown.words(categories="news")
brown_fdist = FreqDist(word.lower() for word in brown_words if word.isalpha())

print(brown_fdist.most_common(20))


## NLTK: A Tiny Text Classifier

Text classification assigns labels to texts. This small example uses hand-written training data and NLTK's Naive Bayes classifier.

This is only a teaching example. Real classification needs more data, train/test splits, evaluation metrics, and careful feature design.


In [ ]:
from nltk.classify import NaiveBayesClassifier

training_data = [
    ("I love this course", "positive"),
    ("This notebook is helpful", "positive"),
    ("Python examples are clear", "positive"),
    ("The lesson is practical", "positive"),
    ("I dislike this task", "negative"),
    ("This explanation is confusing", "negative"),
    ("The exercise is too hard", "negative"),
    ("The setup is frustrating", "negative"),
]


def document_features(sentence):
    words = set(word.lower() for word in word_tokenize(sentence) if word.isalpha())
    return {
        "contains_python": "python" in words,
        "contains_helpful": "helpful" in words,
        "contains_clear": "clear" in words,
        "contains_practical": "practical" in words,
        "contains_confusing": "confusing" in words,
        "contains_hard": "hard" in words,
        "contains_frustrating": "frustrating" in words,
        "contains_love": "love" in words,
        "contains_dislike": "dislike" in words,
    }


featuresets = [(document_features(sentence), label) for sentence, label in training_data]
classifier = NaiveBayesClassifier.train(featuresets)

test_sentences = [
    "This Python notebook is helpful",
    "The setup is confusing",
    "The practical examples are clear",
]

for sentence in test_sentences:
    print(sentence, "->", classifier.classify(document_features(sentence)))

print("\nMost informative features:")
classifier.show_most_informative_features()


# Part 2: spaCy

spaCy uses a pipeline approach:

1. Load a trained language pipeline.
2. Pass raw text to the pipeline.
3. Receive a `Doc` object with tokens and annotations.
4. Inspect linguistic features through object attributes.
5. Add rule-based patterns or custom components when needed.


## spaCy: Load the English Pipeline

We will use `en_core_web_sm`, the small English pipeline. It is compact and good for classroom examples.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
print(nlp.pipe_names)
print(nlp.meta.get("name"), nlp.meta.get("version"))


## spaCy: `Doc`, `Token`, and `Span`

Calling `nlp(text)` returns a `Doc`. A `Doc` is a sequence of `Token` objects and also stores annotations produced by pipeline components.


In [ ]:
doc = nlp(main_text)

print(type(nlp))
print(type(doc))
print(type(doc[0]))
print(type(doc[0:3]))


## spaCy: Tokenization

spaCy tokenization is non-destructive: the original text can be reconstructed from tokens, and character offsets are preserved.


In [ ]:
for token in doc:
    print(f"{token.text:12} idx={token.idx:<3} alpha={token.is_alpha} punct={token.is_punct} stop={token.is_stop}")

print("\nOriginal text preserved:", doc.text == main_text)


## spaCy: Sentence Segmentation

`doc.sents` yields sentence spans.


In [ ]:
for sent in doc.sents:
    print(f"start={sent.start:<2} end={sent.end:<2} text={sent.text}")


## spaCy: Lemmas, POS Tags, and Morphology

spaCy stores common linguistic annotations as token attributes.


In [ ]:
for token in doc:
    print(f"{token.text:12} lemma={token.lemma_:12} pos={token.pos_:6} tag={token.tag_:5} morph={token.morph}")


## spaCy: Dependency Parsing

Dependency parsing identifies grammatical relations between tokens. Each token has a dependency label and a syntactic head.


In [ ]:
for token in doc:
    print(f"{token.text:12} dep={token.dep_:10} head={token.head.text}")


In [ ]:
print("Selected subject/object/prepositional-object relations:")
for token in doc:
    if token.dep_ in {"nsubj", "dobj", "obj", "pobj"}:
        print(f"{token.text:12} {token.dep_:8} -> {token.head.text}")


## spaCy: Noun Chunks

Noun chunks are base noun phrases. They are useful for quick phrase extraction.


In [ ]:
for chunk in doc.noun_chunks:
    print(f"{chunk.text:25} root={chunk.root.text:12} dep={chunk.root.dep_:10} head={chunk.root.head.text}")


## spaCy: Named Entity Recognition

Named Entity Recognition identifies spans such as people, organizations, places, dates, and monetary values.

NER is model-based, so results can be wrong. Always inspect output for your use case.


In [ ]:
news_doc = nlp(news_text)

for ent in news_doc.ents:
    print(f"{ent.text:25} {ent.label_:10} start={ent.start_char:<3} end={ent.end_char}")


## spaCy: Visualizing Entities and Dependencies

spaCy includes `displacy` visualizers. These work in Jupyter and Colab.

Short examples are best because dependency diagrams can become wide.


In [ ]:
from spacy import displacy

short_doc = nlp("Anna joined Acme Robotics in London.")
displacy.render(short_doc, style="ent", jupyter=True)


In [ ]:
displacy.render(short_doc, style="dep", jupyter=True, options={"compact": True})


## spaCy: Rule-Based Matching with `Matcher`

The `Matcher` finds token patterns. It is useful when a rule is clearer than a statistical model.


In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

pattern = [
    {"LOWER": "python"},
    {"LOWER": {"IN": ["course", "notebook", "tools"]}},
]

matcher.add("PYTHON_TERM", [pattern])

matcher_doc = nlp("This Python notebook includes Python tools for the Python course.")
matches = matcher(matcher_doc)

for match_id, start, end in matches:
    label = nlp.vocab.strings[match_id]
    span = matcher_doc[start:end]
    print(label, "->", span.text)


## spaCy: Rule-Based Entities with `EntityRuler`

`EntityRuler` adds entity spans from patterns. It is useful for course-specific terms, product catalogs, known names, or domain dictionaries.


In [ ]:
# Create a fresh pipeline so this cell can be rerun safely.
ruler_nlp = spacy.load("en_core_web_sm")

if "entity_ruler" not in ruler_nlp.pipe_names:
    ruler = ruler_nlp.add_pipe("entity_ruler", before="ner")
else:
    ruler = ruler_nlp.get_pipe("entity_ruler")

ruler.add_patterns([
    {"label": "COURSE", "pattern": "LU Python course"},
    {"label": "TOPIC", "pattern": "language data"},
])

ruler_doc = ruler_nlp("The LU Python course includes examples about language data.")

for ent in ruler_doc.ents:
    print(ent.text, ent.label_)


## spaCy: Batch Processing with `nlp.pipe()`

For many texts, use `nlp.pipe()` instead of calling `nlp()` repeatedly.


In [ ]:
texts = [
    "Anna joined Acme Robotics in London.",
    "Python is useful for language data.",
    "The workshop takes place in Berlin on Monday.",
    "Marie Curie won the Nobel Prize.",
]

for processed_doc in nlp.pipe(texts):
    entities = [(ent.text, ent.label_) for ent in processed_doc.ents]
    print(processed_doc.text)
    print("  entities:", entities)


# Part 3: NLTK and spaCy Side by Side

The two libraries can solve overlapping tasks, but their style is different.

| Question | NLTK | spaCy |
| --- | --- | --- |
| Best teaching role | Classic NLP concepts, corpora, transparent algorithms | Modern NLP pipelines and structured annotations |
| Typical workflow | Call individual functions and classes | Load a pipeline and process text into `Doc` objects |
| Strengths | Corpora, tokenizers, stemmers, taggers, classifiers, WordNet | Speed, production APIs, POS, dependencies, NER, matchers |
| Output style | lists, tuples, trees, frequency distributions | `Doc`, `Token`, `Span`, annotations |
| Main caution | Many functions require separate NLTK data downloads | Results depend on the installed trained pipeline |


## Tokenization Comparison


In [ ]:
comparison_text = "Anna doesn't dislike Python; she teaches NLP in London."

nltk_tokens = word_tokenize(comparison_text)
spacy_doc = nlp(comparison_text)
spacy_tokens = [token.text for token in spacy_doc]

print("NLTK tokens:")
print(nltk_tokens)

print("\nspaCy tokens:")
print(spacy_tokens)


## Lemmatization Comparison

NLTK lemmatization is explicit and often needs a POS argument. spaCy lemmatization comes from the loaded pipeline.


In [ ]:
lemma_words = ["running", "was", "better", "studies", "cars"]

print("NLTK WordNet lemmatizer:")
for word in lemma_words:
    print(f"{word:10} noun={lemmatizer.lemmatize(word):10} verb={lemmatizer.lemmatize(word, pos='v'):10}")

print("\nspaCy lemmas:")
lemma_doc = nlp("running was better studies cars")
for token in lemma_doc:
    print(f"{token.text:10} -> {token.lemma_}")


## Named Entity Comparison

NLTK named entity chunking returns a tree. spaCy stores named entities as spans in `doc.ents`.


In [ ]:
entity_text = "Marie Curie worked in Paris and Google opened an office in Berlin."

print("NLTK named entities:")
nltk_entity_tree = ne_chunk(pos_tag(word_tokenize(entity_text)))
for node in nltk_entity_tree:
    if hasattr(node, "label"):
        entity = " ".join(token for token, tag in node.leaves())
        print(entity, "->", node.label())

print("\nspaCy named entities:")
spacy_entity_doc = nlp(entity_text)
for ent in spacy_entity_doc.ents:
    print(ent.text, "->", ent.label_)


# Part 4: Exercises

Try these tasks after running the demonstration cells.


## Exercise 1: Tokenize and Count

Use `email_text`:

1. Tokenize the text with NLTK.
2. Keep only alphabetic tokens.
3. Lowercase the tokens.
4. Remove English stop words.
5. Print the 10 most common remaining words.


In [ ]:
# Exercise 1 starter code
exercise_tokens = word_tokenize(email_text)

# TODO: filter, lowercase, remove stop words, and count with FreqDist
exercise_tokens[:10]


## Exercise 2: Stemming vs. Lemmatization

Compare stems and lemmas for this list:

```python
["runs", "running", "ran", "studies", "studied", "better", "cars"]
```

Which outputs are dictionary words? Which outputs are useful for grouping word forms?


In [ ]:
# Exercise 2 starter code
exercise_words = ["runs", "running", "ran", "studies", "studied", "better", "cars"]

# TODO: print each word with its Porter stem and WordNet lemma
exercise_words


## Exercise 3: POS Tags

Use NLTK to tag `review_text`, then print only nouns and verbs.


In [ ]:
# Exercise 3 starter code
review_tags = pos_tag(word_tokenize(review_text))

# TODO: print only nouns and verbs
review_tags[:10]


## Exercise 4: spaCy Entities and Noun Chunks

Process `email_text` with spaCy.

1. Print all named entities.
2. Print all noun chunks.
3. Which output is more useful for this email-like text?


In [ ]:
# Exercise 4 starter code
email_doc = nlp(email_text)

# TODO: print entities and noun chunks
email_doc.text


## Exercise 5: spaCy Matcher

Create a matcher pattern that finds:

- `final NLP report`
- `short summary`
- `language data`

Test it on `email_text` and `main_text`.


In [ ]:
# Exercise 5 starter code
exercise_matcher = Matcher(nlp.vocab)

# TODO: add patterns and print matches


## Exercise 6: Mini Classification Challenge

Create at least 12 short labeled sentences: 6 positive and 6 negative. Train an NLTK Naive Bayes classifier and test it on 3 new sentences.

Questions to think about:

- Which words are useful features?
- Does the classifier fail on neutral sentences?
- What happens if a test sentence contains words not seen during training?


In [ ]:
# Exercise 6 starter code
student_training_data = [
    # ("sentence", "label"),
]

# TODO: create features, train classifier, test classifier


# Summary

NLTK and spaCy both support NLP in Python, but they are optimized for different teaching and practical workflows.

Use **NLTK** when you want to understand and demonstrate individual NLP steps:

- tokenization;
- stop word filtering;
- stemming;
- WordNet lemmatization;
- POS tagging;
- chunking;
- corpus exploration;
- simple classifiers.

Use **spaCy** when you want a modern, integrated NLP pipeline:

- fast tokenization and annotation;
- lemmas, POS tags, morphology, and dependencies;
- named entity recognition;
- noun chunks;
- rule-based matching;
- entity rules;
- batch processing.

For real projects, the best choice depends on the task, the language, the required accuracy, processing speed, and how much control you need over each processing step.


# References

Primary library documentation:

- NLTK official documentation: <https://www.nltk.org/>
- NLTK installation: <https://www.nltk.org/install.html>
- NLTK data installation: <https://www.nltk.org/data.html>
- NLTK API reference: <https://www.nltk.org/api/nltk.html>
- NLTK Book, *Natural Language Processing with Python*: <https://www.nltk.org/book/>
- spaCy official site: <https://spacy.io/>
- spaCy installation: <https://spacy.io/usage>
- spaCy models and languages: <https://spacy.io/usage/models>
- spaCy linguistic features: <https://spacy.io/usage/linguistic-features>
- spaCy processing pipelines: <https://spacy.io/usage/processing-pipelines>
- spaCy rule-based matching: <https://spacy.io/usage/rule-based-matching>
- spaCy visualizers: <https://spacy.io/usage/visualizers>

Authoritative NLP references:

- Daniel Jurafsky and James H. Martin, *Speech and Language Processing*, 3rd edition draft: <https://web.stanford.edu/~jurafsky/slp3/>
- Christopher D. Manning, Prabhakar Raghavan, and Hinrich Schuetze, *Introduction to Information Retrieval*, tokenization chapter: <https://nlp.stanford.edu/IR-book/html/htmledition/tokenization-1.html>
- Manning, Raghavan, and Schuetze, stop words section: <https://nlp.stanford.edu/IR-book/html/htmledition/dropping-common-terms-stop-words-1.html>
- Manning, Raghavan, and Schuetze, stemming and lemmatization section: <https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html>
- Princeton WordNet: <https://wordnet.princeton.edu/>

Local source notes used to build this notebook:

- `sources/nlp_overview.md`
- `sources/nlp_nltk.md`
- `sources/nlp_spacy.md`
